# demo_trust

> Phase 14 (`specs/multiagent/phase14/`): what makes trust information pay, and do the fixes scale? Part A completes a 2 x 2 of trust information on the clue board (public or private, free or costly: `record`, `recordPrivate`, `verify`, `verifyPublic`, one scripted liar, N = 4); Part B reruns `request` and `record` at 8, 16 and 32 agents with a quarter of them liars (`TRUST_PLAN`, `SCALE_PLAN`).

Artifacts (gitignored): `results/multiagent/trust/{quick,full}/` (Part A: `runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `trust.png`) and `results/multiagent/scale/{quick,pilot,full}/` (Part B: the same files, `scale.png`). The response cache and spend ledger are Phase 10's (`results/multiagent/llm/`), so one $150 API cap covers all live spend. Part A reuses Phase 13's `baseline`, `record` and `verify` rows (`results/multiagent/fix/full/`); Part B's N = 8 clean `baseline` and `request` cells equal Phase 13's, so they replay its cached replies.

- **Quick** (normal tests): Part A's new arms and Phase 13's A2 arms, and every Part B cell at seed 900, a few turns, with the fake client (which also checks when the schema allows it); outputs identical for 1 and 8 processes.
- **Part A main**, **Part B pilot** and **Part B main** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live; Part A only after `TRUST_PLAN` is frozen and committed; the pilot only after the numpy forecast is committed (`tools/scale_forecast.py`); Part B's main run only after `SCALE_PLAN` is frozen, on the cells the pilot's cost rule allows.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

from explRL.bench_grid import readCSV, writeCSV
from explRL.clue_runs import runClueAny
from explRL.coop_grid import coopHash
from explRL.fix_grid import A2, fixQuick
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.scale_grid import SCALE_PLAN, TRUST_PLAN, costRule, scalePilotGrid, scaleQuick, trustGrid, trustQuick
from explRL.scale_report import figScale, scaleSummaries, trustLogMetrics, trustSummaries, writeScale, writeTrust
from explRL.train import demo_artifact_dir

TRUST, SCALE = 'multiagent/trust', 'multiagent/scale'
WORKERS = min(8, os.cpu_count() or 1)
LLM_WORKERS = 32
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
FIX13 = demo_artifact_dir('full', 'multiagent/fix')          # Phase 13's main run: baseline, record and verify rows
FORECAST = LLM.parents[2] / 'specs/multiagent/phase14/scale_predictions.json'   # the numpy forecast (gate for Part B)


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def readLog(p):
    return [json.loads(x) for x in open(p)] if p.exists() else []


def collect(logs, out):
    "Concatenate the per-run turn logs (sorted by name) into turns.jsonl."
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())


def runCells(mode, sub, grids, spec, workers):
    "Clue-board cells through runWorkload (runClueAny): runs.csv, turns.jsonl, timing.jsonl in results/<sub>/<mode>."
    out = demo_artifact_dir(mode, sub)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload(grids, spec, workers, str(logs), run=runClueAny)
    wall = time.time() - t0
    writeCSV(res['rows'], out / 'runs.csv')
    collect(logs, out)
    res['timing'] = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    return out, res, wall


def finish(out, mode, res, wall, rows, spec, plan, S, write, fig):
    "report.md, the figure and metrics.json (live runs: also the _live copies a cached rerun leaves alone)."
    meta = dict(mode=mode, planHash=coopHash(plan), commit=gitCommit(), model=HAIKU, cap=CAP, wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)),
                calls=int(sum(r['calls'] for r in rows)), spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0)
    write(out / 'report.md', S, meta)
    fig()
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                      timing=res['timing'], plan=plan), indent=2, sort_keys=True, default=str) + '\n')
    if spec.get('kind') == 'live':
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent in all, "
          f"{wall:.0f} s; stopped: {res['stopped']}")
    return meta


def runTrust(mode, grids, spec, rows13=None, logs13=(), workers=None):
    "Part A: the new arms' cells, then the 2 x 2 report; rows13: Phase 13's rows (else the fixA rows of this run)."
    live = spec.get('kind') == 'live'
    out, res, wall = runCells(mode, TRUST, grids, spec, workers or (LLM_WORKERS if live else WORKERS))
    rows = res['rows']
    r13 = rows if rows13 is None else rows13
    S = trustSummaries(r13, rows, trustLogMetrics(list(logs13) + readLog(out / 'turns.jsonl')))
    meta = finish(out, mode, res, wall, rows, spec, TRUST_PLAN, S, writeTrust, lambda: figScale(out, trust=S))
    return rows, S, meta


def runScale(mode, grids, spec, workers=None, study='scaleB'):
    "Part B: the sweep's cells, then the per-N report."
    live = spec.get('kind') == 'live'
    out, res, wall = runCells(mode, SCALE, grids, spec, workers or (LLM_WORKERS if live else WORKERS))
    rows = res['rows']
    S = scaleSummaries(rows, study)
    meta = finish(out, mode, res, wall, rows, spec, SCALE_PLAN, S, writeScale, lambda: figScale(out, scale=S))
    return rows, S, meta


def pilotCost(rows):
    "From the pilot's rows: invalid rate, tokens per call (input incl. cache, output) and cost per call."
    calls = sum(r['calls'] for r in rows) or 1
    tok = sum(r['inputTokens'] + r['cacheReadTokens'] + r['cacheWriteTokens'] for r in rows)
    return dict(invalid=sum(r['invalid'] for r in rows) / calls, inputPerCall=tok / calls,
                outputPerCall=sum(r['outputTokens'] for r in rows) / calls, costPerCall=sum(r['cost'] for r in rows) / calls)


def quickTrust():
    "Part A's new arms and Phase 13's A2 arms (study fixA, seeds 900-901, 6 turns)."
    return trustQuick() + [c for c in fixQuick() if c['level'].split('|')[0] in A2 and c['N'] == 4]

## Quick: every cell kind with the fake client (pipeline check)

In [ ]:
t0 = time.time()
rows, S, _ = runTrust('quick', [('trust', quickTrust())], dict(kind='fake'))
srows, T, _ = runScale('quick', [('scale', scaleQuick())], dict(kind='fake'))
assert time.time() - t0 < 120
tq, sq = demo_artifact_dir('quick', TRUST), demo_artifact_dir('quick', SCALE)
for d, fig in ((tq, 'trust.png'), (sq, 'scale.png')):
    for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md', fig):
        assert (d / f).stat().st_size > 0, (d, f)
read = lambda: tuple((d / f).read_bytes() for d in (tq, sq) for f in ('runs.csv', 'turns.jsonl'))
ref = read()
for w, pooled in ((1, True), (8, False)):
    runTrust('quick', [('trust', quickTrust())], dict(kind='fake', pooled=pooled), workers=w)
    runScale('quick', [('scale', scaleQuick())], dict(kind='fake', pooled=pooled), workers=w)
    assert read() == ref, w
# every part is reported; the new settings show in the observations
assert set(S['arms']) == {'record', 'recordPrivate', 'verify', 'verifyPublic'} and set(S['effects']) == {'publicity', 'cost', 'difference'}
assert {r['level'] for r in rows if r['study'] == 'trustA'} == {f'{a}{x}' for a in ('recordPrivate', 'verifyPublic') for x in ('', '|liar')}
recs = readLog(tq / 'turns.jsonl')
lv = lambda x: x['key'].split('|')[2].split('|')[0]
assert all(', record ' in x['obs'] or 'Clues shared on the board: none' in x['obs'] for x in recs if lv(x) in ('record', 'recordPrivate'))
assert any('Checked by agent' in x['obs'] for x in recs if lv(x) == 'verifyPublic')
assert not any('Checked by agent' in x['obs'] for x in recs if lv(x) != 'verifyPublic')
m = trustLogMetrics(recs)
assert sum(v['checks'] for k, v in m.items() if k.split('|')[2].startswith('verifyPublic')) > 0
assert set(T['request']) == {f'request|{N}' for N in (8, 16, 32)} and set(T['record']) == {f'record|{N}' for N in (8, 16, 32)}
assert {(r['N'], r['nLiar']) for r in srows if r['level'].endswith('|liar')} == {(8, 2), (16, 4), (32, 8)}
for d in (tq, sq):
    for p in d.iterdir():
        if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
            assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(tq / 'report.md').read()[:1500])
print(open(sq / 'report.md').read()[:1500])

## Part A main (`--cell-timeout 0 --flags llm`, live, about 5 600 calls, about $1.70)

`trustGrid` (`recordPrivate` and `verifyPublic` at N = 4, with one liar and without, seeds 900-909, 40 turns); `record`, `verify` and `baseline` come from Phase 13's main run (`results/multiagent/fix/full/`). `TRUST_PLAN` must be frozen and committed first.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert TRUST_PLAN['frozen'], 'freeze and commit TRUST_PLAN before the main run'
    assert (FIX13 / 'runs.csv').exists(), "Part A needs Phase 13's main run (results/multiagent/fix/full)"
    rows, S, meta = runTrust('full', [('trustA', trustGrid())], LIVE_SPEC, readCSV(FIX13 / 'runs.csv'), readLog(FIX13 / 'turns.jsonl'))
    print(open(demo_artifact_dir('full', TRUST) / 'report.md').read()[:4000])

## Part B pilot (`--flags llm`, live, 2 240 calls, about $1-2)

N = 32, `baseline`, clean and with 8 liars, seed 900 (the main run replays these replies). Only after the numpy forecast is committed. AC-4: invalid replies under 5% at N = 32; tokens and cost per call measured; the cost rule (`SCALE_PLAN['costRule']`) applied as written: the main grid within $40, else N = 32 on seeds 900-902, else stop and ask.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert FORECAST.exists(), 'commit the numpy forecast (tools/scale_forecast.py run) before the pilot'
    prows, _, _ = runScale('pilot', [('pilot', scalePilotGrid())], LIVE_SPEC, study='scalePilot')
    pc = pilotCost(prows)
    for r in prows:
        print(f"{r['level']:>14} N {r['N']}: invalid {r['invalid'] / max(r['calls'], 1):.3f}, pooled share {r['pooledShare']:.2f}, "
              f"honest welfare {r['welfareHonest']:.2f}")
    print(f"invalid rate at N 32: {pc['invalid']:.3f} (band < {SCALE_PLAN['bands']['invalid']}); tokens per call: "
          f"{pc['inputPerCall']:.0f} in, {pc['outputPerCall']:.0f} out; cost per call ${pc['costPerCall']:.5f}")
    rule = costRule(pc['costPerCall'])
    print(f"projected main cost: ${rule['projected']['full']:.2f} (N 32 on seeds 900-902: ${rule['projected']['cut']:.2f}); "
          f"budget ${SCALE_PLAN['costRule']['budget']:.0f}; cost rule: {rule['verdict']}")

## Part B main (`--cell-timeout 0 --flags llm`, live, about 33 000 new calls, up to $40)

`scaleGrid` (N = 8, 16, 32; clean `baseline` and `request`; `baseline` and `record` with N // 4 liars; seeds 900-904, 40 turns) on the cells the pilot's cost rule allows (N = 32 cut to seeds 900-902 if the full grid projects above $40; nothing if the cut one does too: stop and ask). `SCALE_PLAN` must be frozen and committed first.

In [ ]:
#| llm
def scaleMain():
    "The cells the cost rule allows, from the pilot's measured cost per call (None: stop and ask)."
    prows = readCSV(demo_artifact_dir('pilot', SCALE) / 'runs.csv')
    rule = costRule(pilotCost(prows)['costPerCall'])
    print(f"cost rule: {rule['verdict']} (projected ${rule['projected']['full']:.2f}, cut ${rule['projected']['cut']:.2f})")
    return rule['cells']


if not LIVE or not (demo_artifact_dir('pilot', SCALE) / 'runs.csv').exists():
    print('skipped: needs EXPLRL_LLM_LIVE=1 and the pilot')
else:
    assert SCALE_PLAN['frozen'], 'freeze and commit SCALE_PLAN before the main run'
    cells = scaleMain()
    if cells is None:
        print('the cost rule says stop: ask before running Part B')
    else:
        srows, T, smeta = runScale('full', [('scaleB', cells)], LIVE_SPEC)
        print(open(demo_artifact_dir('full', SCALE) / 'report.md').read()[:4000])

## Cached reruns (`--flags llm`, after the main runs)

Every reply comes from the cache: `runs.csv` and `turns.jsonl` byte-identical, nothing spent (AC-6).

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 (needs the main runs)')
else:
    for sub, rerun in ((TRUST, lambda: runTrust('full', [('trustA', trustGrid())], LIVE_SPEC, readCSV(FIX13 / 'runs.csv'),
                                                 readLog(FIX13 / 'turns.jsonl'))),
                       (SCALE, lambda: (cells := scaleMain()) is not None and runScale('full', [('scaleB', cells)], LIVE_SPEC))):
        files = [demo_artifact_dir('full', sub) / f for f in ('runs.csv', 'turns.jsonl')]
        if not files[0].exists():
            print(f'{sub}: skipped, no main run')
            continue
        before, spent = [p.read_bytes() for p in files], Budget(CAP, LIVE_SPEC['ledger']).spent()
        rerun()
        assert [p.read_bytes() for p in files] == before and Budget(CAP, LIVE_SPEC['ledger']).spent() == spent, sub
        print(f'{sub}: cached rerun byte-identical, free')